In [1]:
import requests
from bs4 import BeautifulSoup
from IPython.display import display, HTML

def fetch_allocine_movies():
    """Scrape la liste des films à l'affiche sur AlloCiné."""
    url = "https://www.allocine.fr/films/"
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
    }

    try:
        response = requests.get(url, headers=headers, timeout=10)
        response.raise_for_status()
    except requests.RequestException as e:
        print(f"Erreur HTTP/Connexion : {e}")
        return []

    soup = BeautifulSoup(response.text, "html.parser")
    movies_cards = soup.find_all("div", class_="card entity-card entity-card-list cf")

    movies_list = []
    for card in movies_cards:
        info = parse_movie_card(card)
        if info:
            movies_list.append(info)

    return movies_list

def parse_movie_card(card):
    """Extrait proprement les données d'une carte de film."""
    try:
        title_elem = card.find("a", class_="meta-title-link")
        title = title_elem.text.strip() if title_elem else "Titre inconnu"

        director_elem = card.find("div", class_="meta-body-item meta-body-direction")
        director = director_elem.text.strip().replace("De", "").strip() if director_elem else "Non spécifié"

        synopsis_elem = card.find("div", class_="content-txt")
        synopsis = synopsis_elem.text.strip() if synopsis_elem else "Pas de synopsis"

        img_elem = card.find("img", class_="thumbnail-img")
        img_url = ""
        if img_elem:
            img_url = img_elem.get("data-src") or img_elem.get("src", "")

        date_elem = card.find("span", class_="date")
        release_date = date_elem.text.strip() if date_elem else "Date inconnue"

        return {
            "Titre": title,
            "Réalisateur": director,
            "Date de sortie": release_date,
            "Synopsis": synopsis,
            "Affiche": img_url
        }
    except Exception as err:
        return None

def render_html_grid(movies_list):
    """Affiche les films sous forme de cartes HTML dans un Notebook Jupyter."""
    if not movies_list:
        print("Aucune donnée à afficher.")
        return

    css_styles = """
    <style>
        .movies-grid {
            display: grid;
            grid-template-columns: repeat(auto-fill, minmax(280px, 1fr));
            gap: 20px;
            padding: 10px;
            font-family: 'Segoe UI', Tahoma, Geneva, Verdana, sans-serif;
        }
        .movie-card {
            border: 1px solid #e0e0e0;
            border-radius: 10px;
            overflow: hidden;
            box-shadow: 0 4px 6px rgba(0,0,0,0.1);
            background: #1a1a1a;
            display: flex;
            flex-direction: column;
        }
        .movie-card img {
            width: 100%;
            height: 380px;
            object-fit: cover;
        }
        .movie-details {
            padding: 15px;
            display: flex;
            flex-direction: column;
            flex-grow: 1;
        }
        .movie-title {
            font-size: 1.1em;
            font-weight: bold;
            color: #ffffff;
            margin-bottom: 5px;
        }
        .movie-meta {
            font-size: 0.85em;
            color: #ffffff;
            margin-bottom: 10px;
        }
        .movie-synopsis {
            font-size: 0.85em;
            color: #ffffff;
            line-height: 1.4;
            display: -webkit-box;
            -webkit-line-clamp: 4;
            -webkit-box-orient: vertical;
            overflow: hidden;
        }
    </style>
    """

    cards_html = ""
    for movie in movies_list:
        cards_html += f"""
        <div class="movie-card">
            <img src="{movie['Affiche']}" alt="{movie['Titre']}">
            <div class="movie-details">
                <div class="movie-title">{movie['Titre']}</div>
                <div class="movie-meta"><b>Par:</b> {movie['Réalisateur']} | <b>Sortie:</b> {movie['Date de sortie']}</div>
                <div class="movie-synopsis">{movie['Synopsis']}</div>
            </div>
        </div>
        """

    full_html = f"{css_styles}<div class='movies-grid'>{cards_html}</div>"
    display(HTML(full_html))

# Exécution
movies = fetch_allocine_movies()
render_html_grid(movies)